# Bayesian LMM Analysis: Periodic (Π) and Aperiodic (Ξ) EEG Components

Fits Bayesian Linear Mixed-Effects Models using PyMC to estimate
intervention effects on EEG oscillatory energy across anatomical regions.

**Outcome variable:** log-RMS amplitude (dB) per electrode, averaged by region.

**Models:**
- PRE → POST: CTRL vs SHAM vs EXP (3-group comparison)
- POST → SEG: SHAM vs EXP (protective-effect window)

Run the **Periodic** section first, then the **Aperiodic** section.

In [ ]:
# ╔═══════════════╗
# ║  Imports      ║
# ╚═══════════════╝

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D
from matplotlib.font_manager import FontProperties
from matplotlib.gridspec import GridSpec

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  EEG data processing functions           ║
# ╚══════════════════════════════════════════╝

def calculate_subject_rms(df, channels):
    """sqrt(mean(x²)) across all rows, grouped by subject."""
    df_sub = df[['id'] + channels].copy()
    df_sub[channels] = df_sub[channels] ** 2
    return df_sub.groupby('id')[channels].mean().apply(np.sqrt)  # (subjects x channels)

def compute_condition_rms(df, cond, channels, instants=["PRE", "POST"]):
    """Per-subject RMS for Exp and Control groups at each instant, for one condition."""
    rms = {cond: {instants[0]: {}, instants[1]: {}}}
    for ins in instants:
        df_ins = df[(df['cond'] == cond) & (df['instant'] == ins)]
        rms[cond][ins]['Exp']     = calculate_subject_rms(df_ins[df_ins['group'] == 'Exp'],     channels)
        rms[cond][ins]['Control'] = calculate_subject_rms(df_ins[df_ins['group'] == 'Control'], channels)
    return rms

def get_stacked_data(rms_final, instants=["PRE", "POST"]):
    """RMS dict -> long DataFrame (subject x channel) with Delta, Region, Condition, Real_Group."""
    electrode_to_region = {ch: reg for reg, chs in eeg_regions.items() for ch in chs}
    all_stacked = []

    for cond in rms_final:
        df_pre  = pd.concat([rms_final[cond][instants[0]]['Exp'], rms_final[cond][instants[0]]['Control']])
        df_post = pd.concat([rms_final[cond][instants[1]]['Exp'], rms_final[cond][instants[1]]['Control']])

        df_pre.index  = df_pre.index.astype(int)
        df_post.index = df_post.index.astype(int)
        common_ids = df_pre.index.intersection(df_post.index)

        pre_stacked  = df_pre.loc[common_ids].stack().reset_index()
        post_stacked = df_post.loc[common_ids].stack().reset_index()
        pre_stacked.columns  = ['id', 'Channel', 'RMS_' + instants[0]]
        post_stacked.columns = ['id', 'Channel', 'RMS_' + instants[1]]

        df_long = pre_stacked.copy()
        df_long['RMS_' + instants[1]] = post_stacked['RMS_' + instants[1]]
        df_long['Delta']    = df_long['RMS_' + instants[1]] - df_long['RMS_' + instants[0]]
        df_long['Delta_dB'] = (20 * np.log10(df_long['RMS_' + instants[1]])
                             - 20 * np.log10(df_long['RMS_' + instants[0]]))
        df_long['Region']    = df_long['Channel'].map(electrode_to_region)
        df_long['Condition'] = cond

        exp_ids = rms_final[cond][instants[0]]['Exp'].index.astype(int)
        df_long['Real_Group'] = df_long['id'].apply(
            lambda idx: 'Sham' if idx in sham_ids else ('Experimental' if idx in exp_ids else 'Control')
        )
        all_stacked.append(df_long)

    return pd.concat(all_stacked)

def get_long_format(df_stacked, instants=["PRE", "POST"]):
    """Channel-level stacked data -> region-averaged long format with log_RMS, ready for LMM."""
    df_pre_long = df_stacked[['id', 'Channel', 'RMS_' + instants[0], 'Region', 'Condition', 'Real_Group']].copy()
    df_pre_long = df_pre_long.rename(columns={'RMS_' + instants[0]: 'RMS'})
    df_pre_long['Instant'] = instants[0]

    df_post_long = df_stacked[['id', 'Channel', 'RMS_' + instants[1], 'Region', 'Condition', 'Real_Group']].copy()
    df_post_long = df_post_long.rename(columns={'RMS_' + instants[1]: 'RMS'})
    df_post_long['Instant'] = instants[1]

    df_region_long = (
        pd.concat([df_pre_long, df_post_long])
        .reset_index(drop=True)
        .groupby(['id', 'Region', 'Condition', 'Real_Group', 'Instant'], as_index=False)['RMS']
        .mean()
    )
    df_region_long = df_region_long[df_region_long['RMS'] > 0].copy()
    df_region_long['log_RMS'] = 20 * np.log10(df_region_long['RMS'])
    return df_region_long

def get_region_deltas(stacked, cond):
    per_subject = (
        stacked[cond]
        .groupby(['id', 'Real_Group', 'Region'], as_index=False)['Delta_dB']
        .median()
    )
    return (
        per_subject
        .groupby(['Real_Group', 'Region'], as_index=False)['Delta_dB']
        .mean()
        .rename(columns={'Delta_dB': 'Delta'})  # keeps heatmap interface compatible
    )

In [ ]:
# ╔══════════════════════════════════════════╗
# ║  Modelling utilities                     ║
# ╚══════════════════════════════════════════╝

REGION_MAP = {"Frontal": 0, "Central": 1, "Occipital": 2, "Parietal": 3, "Temporal": 4}

# ── Pipeline ──────────────────────────────────────────────────────────────────

def build_pipeline(df_all, channels, instants):
    rms_final = {}
    for cond in ['OA', 'OC']:
        rms_final.update(compute_condition_rms(df_all, cond=cond, channels=channels, instants=instants))
    df_stacked = get_stacked_data(rms_final, instants=instants)
    df_long    = get_long_format(df_stacked, instants=instants)
    return (
        {cond: df_long[df_long['Condition'] == cond].copy()       for cond in ['OA', 'OC']},
        {cond: df_stacked[df_stacked['Condition'] == cond].copy() for cond in ['OA', 'OC']},
    )

def encode_and_filter(df, instant_map, group_map, groups=None):
    """Encode Instant/Group/Region as integer indices; optionally filter to a subset of groups."""
    df = df.copy()
    string_cols = df.select_dtypes(include="string").columns
    df[string_cols] = df[string_cols].astype(object)
    if groups is not None:
        df = df[df['Real_Group'].isin(groups)].copy()
    df['instant_idx'] = df['Instant'].map(instant_map)
    df['group_idx']   = df['Real_Group'].map(group_map)
    df['region_idx']  = df['Region'].map(REGION_MAP)
    return df

# ── Array preparation ─────────────────────────────────────────────────────────

def _region_indicators(region):
    """Binary indicator arrays for non-reference regions (Frontal = reference)."""
    return {
        "is_central":   (region == 1).astype(int),
        "is_occipital": (region == 2).astype(int),
        "is_parietal":  (region == 3).astype(int),
        "is_temporal":  (region == 4).astype(int),
    }

def prepare_pre_post_arrays(df):
    """
    Numpy arrays for a PRE->POST LMM.
    Reference: Control group, PRE instant, Frontal region.
    """
    subjects, subject_idx = np.unique(df['id'], return_inverse=True)
    group   = df['group_idx'].values
    instant = df['instant_idx'].values
    region  = df['region_idx'].values
    is_exp  = (group == 1).astype(int)
    is_sham = (group == 2).astype(int)
    is_post = instant
    return {
        "y":            df['log_RMS'].values,
        "subject_idx":  subject_idx,
        "n_subjects":   len(subjects),
        "is_exp":       is_exp,
        "is_sham":      is_sham,
        "is_post":      is_post,
        "is_exp_post":  is_exp * is_post,
        "is_sham_post": is_sham * is_post,
        **_region_indicators(region),
    }

def prepare_post_fu_arrays(df):
    """
    Numpy arrays for a POST->SEG LMM.
    Reference: Sham group, POST instant, Frontal region.
    """
    subjects, subject_idx = np.unique(df['id'], return_inverse=True)
    group   = df['group_idx'].values   # Sham=0, Experimental=1
    instant = df['instant_idx'].values
    region  = df['region_idx'].values
    is_exp  = group                    # directly usable as indicator
    is_fu   = instant
    return {
        "y":           df['log_RMS'].values,
        "subject_idx": subject_idx,
        "n_subjects":  len(subjects),
        "is_exp":      is_exp,
        "is_fu":       is_fu,
        "is_exp_fu":   is_exp * is_fu,
        **_region_indicators(region),
    }

# ── Model builders ────────────────────────────────────────────────────────────

def build_pre_post_lmm(arrays):
    a = arrays
    with pm.Model() as model:
        # Fixed effects
        intercept      = pm.Normal("intercept",      mu=20.0, sigma=10.0)
        beta_exp       = pm.Normal("beta_exp",        mu=0,    sigma=2.0)
        beta_sham      = pm.Normal("beta_sham",       mu=0,    sigma=2.0)
        beta_post      = pm.Normal("beta_post",       mu=0,    sigma=2.0)
        beta_exp_post  = pm.Normal("beta_exp_post",   mu=0,    sigma=2.0)
        beta_sham_post = pm.Normal("beta_sham_post",  mu=0,    sigma=2.0)

        # Region effects — non-centred (Frontal = reference)
        sigma_region       = pm.HalfNormal("sigma_region", sigma=2.0)
        beta_central_raw   = pm.Normal("beta_central_raw",   mu=0, sigma=1.0)
        beta_occipital_raw = pm.Normal("beta_occipital_raw", mu=0, sigma=1.0)
        beta_parietal_raw  = pm.Normal("beta_parietal_raw",  mu=0, sigma=1.0)
        beta_temporal_raw  = pm.Normal("beta_temporal_raw",  mu=0, sigma=1.0)
        beta_central   = pm.Deterministic("beta_central",   beta_central_raw   * sigma_region)
        beta_occipital = pm.Deterministic("beta_occipital", beta_occipital_raw * sigma_region)
        beta_parietal  = pm.Deterministic("beta_parietal",  beta_parietal_raw  * sigma_region)
        beta_temporal  = pm.Deterministic("beta_temporal",  beta_temporal_raw  * sigma_region)

        # Subject random effects — non-centred
        sigma_subj = pm.HalfNormal("sigma_subj", sigma=2.0)
        u_subj_raw = pm.Normal("u_subj_raw", mu=0, sigma=1.0, shape=a["n_subjects"])
        u_subj     = pm.Deterministic("u_subj", u_subj_raw * sigma_subj)

        # Residual
        sigma_eps = pm.HalfNormal("sigma_eps", sigma=2.0)

        mu = (intercept
              + beta_exp        * a["is_exp"]
              + beta_sham       * a["is_sham"]
              + beta_post       * a["is_post"]
              + beta_exp_post   * a["is_exp_post"]
              + beta_sham_post  * a["is_sham_post"]
              + beta_central    * a["is_central"]
              + beta_occipital  * a["is_occipital"]
              + beta_parietal   * a["is_parietal"]
              + beta_temporal   * a["is_temporal"]
              + u_subj[a["subject_idx"]])
        pm.Normal("likelihood", mu=mu, sigma=sigma_eps, observed=a["y"])
    return model

def build_post_fu_lmm(arrays):
    a = arrays
    with pm.Model() as model:
        # Fixed effects
        intercept   = pm.Normal("intercept",   mu=20.0, sigma=10.0)
        beta_exp    = pm.Normal("beta_exp",    mu=0,    sigma=2.0)
        beta_fu     = pm.Normal("beta_fu",     mu=0,    sigma=2.0)
        beta_exp_fu = pm.Normal("beta_exp_fu", mu=0,    sigma=2.0)

        # Region effects — non-centred (Frontal = reference)
        sigma_region       = pm.HalfNormal("sigma_region", sigma=2.0)
        beta_central_raw   = pm.Normal("beta_central_raw",   mu=0, sigma=1.0)
        beta_occipital_raw = pm.Normal("beta_occipital_raw", mu=0, sigma=1.0)
        beta_parietal_raw  = pm.Normal("beta_parietal_raw",  mu=0, sigma=1.0)
        beta_temporal_raw  = pm.Normal("beta_temporal_raw",  mu=0, sigma=1.0)
        beta_central   = pm.Deterministic("beta_central",   beta_central_raw   * sigma_region)
        beta_occipital = pm.Deterministic("beta_occipital", beta_occipital_raw * sigma_region)
        beta_parietal  = pm.Deterministic("beta_parietal",  beta_parietal_raw  * sigma_region)
        beta_temporal  = pm.Deterministic("beta_temporal",  beta_temporal_raw  * sigma_region)

        # Subject random effects — non-centred
        sigma_subj = pm.HalfNormal("sigma_subj", sigma=2.0)
        u_subj_raw = pm.Normal("u_subj_raw", mu=0, sigma=1.0, shape=a["n_subjects"])
        u_subj     = pm.Deterministic("u_subj", u_subj_raw * sigma_subj)

        # Residual
        sigma_eps = pm.HalfNormal("sigma_eps", sigma=2.0)

        mu = (intercept
              + beta_exp        * a["is_exp"]
              + beta_fu         * a["is_fu"]
              + beta_exp_fu     * a["is_exp_fu"]
              + beta_central    * a["is_central"]
              + beta_occipital  * a["is_occipital"]
              + beta_parietal   * a["is_parietal"]
              + beta_temporal   * a["is_temporal"]
              + u_subj[a["subject_idx"]])
        pm.Normal("likelihood", mu=mu, sigma=sigma_eps, observed=a["y"])
    return model

# ── Results ───────────────────────────────────────────────────────────────────

def extract_pre_post_posteriors(trace):
    """Marginal posterior distributions per group x instant from a PRE->POST trace."""
    p              = trace.posterior
    intercept      = p["intercept"].values.flatten()
    beta_exp       = p["beta_exp"].values.flatten()
    beta_sham      = p["beta_sham"].values.flatten()
    beta_post      = p["beta_post"].values.flatten()
    beta_exp_post  = p["beta_exp_post"].values.flatten()
    beta_sham_post = p["beta_sham_post"].values.flatten()
    return {
        "ctrl_pre":  intercept,
        "exp_pre":   intercept + beta_exp,
        "sham_pre":  intercept + beta_sham,
        "ctrl_post": intercept + beta_post,
        "exp_post":  intercept + beta_exp  + beta_post + beta_exp_post,
        "sham_post": intercept + beta_sham + beta_post + beta_sham_post,
    }

def extract_post_fu_posteriors(trace):
    """Marginal posterior distributions per group x instant from a POST->SEG trace."""
    p           = trace.posterior
    intercept   = p["intercept"].values.flatten()
    beta_exp    = p["beta_exp"].values.flatten()
    beta_fu     = p["beta_fu"].values.flatten()
    beta_exp_fu = p["beta_exp_fu"].values.flatten()
    return {
        "sham_post": intercept,
        "exp_post":  intercept + beta_exp,
        "sham_fu":   intercept + beta_fu,
        "exp_fu":    intercept + beta_exp + beta_fu + beta_exp_fu,
    }

def run_and_print_bayesian_lmm_hdi(trace, condition_name):
    print("=" * 75)
    print(f" BAYESIAN LMM & ICC (95% HDI) - CONDITION: {condition_name} ".center(75, "="))
    print("=" * 75)

    def compute_hdi_summary(trace, var_names, hdi_prob=0.95):
        rows = []
        for var in var_names:
            if var not in trace.posterior:
                continue
            samples = trace.posterior[var].values.flatten()
            hdi     = az.hdi(samples, prob=hdi_prob)
            df_rhat = az.summary(trace, var_names=[var], round_to=3)
            alpha   = (1 - hdi_prob) / 2
            rows.append({
                "variable":                  var,
                "mean":                      round(float(np.mean(samples)), 3),
                "sd":                        round(float(np.std(samples)),  3),
                f"hdi_{alpha*100:.1f}%":     round(float(hdi[0]), 3),
                f"hdi_{(1-alpha)*100:.1f}%": round(float(hdi[1]), 3),
                "r_hat":                     round(float(df_rhat["r_hat"].values[0]), 3),
            })
        return pd.DataFrame(rows).set_index("variable")

    print("\n[FIXED EFFECTS SUMMARY]")
    fixed_vars = [
        "intercept", "beta_exp", "beta_sham",
        "beta_post", "beta_fu",
        "beta_exp_post", "beta_sham_post", "beta_exp_fu",
        "beta_central", "beta_occipital", "beta_parietal", "beta_temporal",
    ]
    print(compute_hdi_summary(trace, [v for v in fixed_vars if v in trace.posterior]))

    print("\n[RANDOM EFFECTS SCALES (SIGMAS)]")
    print(compute_hdi_summary(trace, ["sigma_subj", "sigma_region", "sigma_eps"]))

    print("\n[VARIANCE COMPONENTS & INTRA-CLASS CORRELATION]")
    var_subj    = trace.posterior["sigma_subj"].values.flatten() ** 2
    var_eps     = trace.posterior["sigma_eps"].values.flatten()  ** 2
    icc_samples = var_subj / (var_subj + var_eps)
    icc_hdi     = az.hdi(icc_samples, prob=0.95)
    print(f"  • Between-subject variance (sigma2_u) : {np.mean(var_subj):.4f}")
    print(f"  • Residual variance (sigma2_e)        : {np.mean(var_eps):.4f}")
    print(f"  • ICC                                 : {np.mean(icc_samples):.4f}  "
          f"(95% HDI: [{icc_hdi[0]:.4f}, {icc_hdi[1]:.4f}])")
    print("\n" + "=" * 75 + "\n")

In [ ]:
# ╔══════════════════════════════════╗
# ║  Constants                       ║
# ╚══════════════════════════════════╝

sham_ids = [
     1,  4,  7,  9, 11, 13, 14, 15,
    17, 18, 19, 24, 25, 27, 28,
    31, 35, 40, 41, 42, 43, 44,
    47, 48, 49, 50, 52, 61, 63, 65, 105,
]

ids_to_exclude = [19, 51, 23, 5, 27, 71, 87]

COMPONENTS = {
    "pi": {
        "channels": [
            'P8_pi', 'T8_pi', 'F8_pi', 'F4_pi', 'C4_pi',
            'P4_pi', 'Fp2_pi', 'Fp1_pi', 'Fz_pi', 'Cz_pi',
            'O1_pi', 'Oz_pi', 'O2_pi', 'Pz_pi', 'P3_pi',
            'C3_pi', 'F3_pi', 'F7_pi', 'T7_pi', 'P7_pi',
        ],
        "regions": {
            "Frontal":   ["Fp1_pi", "Fp2_pi", "F3_pi", "F4_pi", "F7_pi", "F8_pi", "Fz_pi"],
            "Central":   ["C3_pi", "C4_pi", "Cz_pi"],
            "Parietal":  ["P3_pi", "P4_pi", "P7_pi", "P8_pi", "Pz_pi"],
            "Occipital": ["O1_pi", "O2_pi", "Oz_pi"],
            "Temporal":  ["T7_pi", "T8_pi"],
        }
    },
    "xi": {
        "channels": [
            'P8_xi', 'T8_xi', 'F8_xi', 'F4_xi', 'C4_xi',
            'P4_xi', 'Fp2_xi', 'Fp1_xi', 'Fz_xi', 'Cz_xi',
            'O1_xi', 'Oz_xi', 'O2_xi', 'Pz_xi', 'P3_xi',
            'C3_xi', 'F3_xi', 'F7_xi', 'T7_xi', 'P7_xi',
        ],
        "regions": {
            "Frontal":   ["Fp1_xi", "Fp2_xi", "F3_xi", "F4_xi", "F7_xi", "F8_xi", "Fz_xi"],
            "Central":   ["C3_xi", "C4_xi", "Cz_xi"],
            "Parietal":  ["P3_xi", "P4_xi", "P7_xi", "P8_xi", "Pz_xi"],
            "Occipital": ["O1_xi", "O2_xi", "Oz_xi"],
            "Temporal":  ["T7_xi", "T8_xi"],
        }
    }
}

pre_post_posteriors = {}   # pre_post_posteriors["pi"] and ["xi"]
post_fu_posteriors  = {}   # post_fu_posteriors["pi"]  and ["xi"]
stacked_pp = {}            # stacked_pp["pi"] and ["xi"]
stacked_ps = {}            # stacked_ps["pi"] and ["xi"]

In [ ]:
# ╔══════════════════════════════════╗
# ║  Load & filter data              ║
# ╚══════════════════════════════════╝

def load_data(component: str) -> pd.DataFrame:
    cfg  = COMPONENTS[component]
    cols = ['cond', 'instant', 'group', 'id'] + cfg["channels"]
    df   = pd.read_parquet("full_eeg_reconstructed.parquet", columns=cols)
    df   = df[~df['id'].isin(ids_to_exclude)].copy()
    print(f"[{component}] IDs excluded: {ids_to_exclude}")
    print(f"[{component}] Unique subjects remaining: {df['id'].nunique()}")
    return df

# Periodic (Π)

In [ ]:
COMPONENT   = "pi"
channels    = COMPONENTS[COMPONENT]["channels"]
eeg_regions = COMPONENTS[COMPONENT]["regions"]
df_all      = load_data(COMPONENT)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 — PRE->POST: pipeline & array preparation    ║
# ╚══════════════════════════════════════════════════════╝

INSTANT_MAP_PP = {"PRE": 0, "POST": 1}
GROUP_MAP_3    = {"Control": 0, "Experimental": 1, "Sham": 2}  # 3-group model

dfs_pp, stacked   = build_pipeline(df_all, channels, instants=["PRE", "POST"])
stacked_pp[COMPONENT] = stacked

df_OA_pp = encode_and_filter(dfs_pp['OA'], INSTANT_MAP_PP, GROUP_MAP_3)
df_OC_pp = encode_and_filter(dfs_pp['OC'], INSTANT_MAP_PP, GROUP_MAP_3)

# Optional sanity check:
# print(df_OA_pp.head(10)); print(df_OA_pp.groupby('id').size().unique())

arrays_OA_pp = prepare_pre_post_arrays(df_OA_pp)
arrays_OC_pp = prepare_pre_post_arrays(df_OC_pp)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 — Build & sample: Eyes Open PRE->POST        ║
# ╚══════════════════════════════════════════════════════╝

eo_pre_post_lmm = build_pre_post_lmm(arrays_OA_pp)

with eo_pre_post_lmm:
    eo_pre_post_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — Build & sample: Eyes Closed PRE->POST      ║
# ╚══════════════════════════════════════════════════════╝

ec_pre_post_lmm = build_pre_post_lmm(arrays_OC_pp)

with ec_pre_post_lmm:
    ec_pre_post_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 9 — PRE->POST results                          ║
# ╚══════════════════════════════════════════════════════╝

run_and_print_bayesian_lmm_hdi(eo_pre_post_trace, "Eyes Open (OA)")
run_and_print_bayesian_lmm_hdi(ec_pre_post_trace, "Eyes Closed (OC)")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 10 — Extract PRE->POST posteriors              ║
# ╚══════════════════════════════════════════════════════╝

post_eo = extract_pre_post_posteriors(eo_pre_post_trace)
ctrl_pre_eo,  exp_pre_eo,  sham_pre_eo  = post_eo["ctrl_pre"],  post_eo["exp_pre"],  post_eo["sham_pre"]
ctrl_post_eo, exp_post_eo, sham_post_eo = post_eo["ctrl_post"], post_eo["exp_post"], post_eo["sham_post"]

post_ec = extract_pre_post_posteriors(ec_pre_post_trace)
ctrl_pre_ec,  exp_pre_ec,  sham_pre_ec  = post_ec["ctrl_pre"],  post_ec["exp_pre"],  post_ec["sham_pre"]
ctrl_post_ec, exp_post_ec, sham_post_ec = post_ec["ctrl_post"], post_ec["exp_post"], post_ec["sham_post"]

pre_post_posteriors[COMPONENT] = {
    "Eyes Open":   {"ctrl_pre": ctrl_pre_eo,  "exp_pre": exp_pre_eo,  "sham_pre": sham_pre_eo,
                    "ctrl_post": ctrl_post_eo, "exp_post": exp_post_eo, "sham_post": sham_post_eo},
    "Eyes Closed": {"ctrl_pre": ctrl_pre_ec,  "exp_pre": exp_pre_ec,  "sham_pre": sham_pre_ec,
                    "ctrl_post": ctrl_post_ec, "exp_post": exp_post_ec, "sham_post": sham_post_ec},
}

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 11 — POST->SEG: pipeline & array preparation   ║
# ╚══════════════════════════════════════════════════════╝

INSTANT_MAP_PS = {"POST": 0, "SEGUIMIENTO": 1}
GROUP_MAP_2    = {"Sham": 0, "Experimental": 1}  # 2-group model

dfs_ps, stacked   = build_pipeline(df_all, channels, instants=["POST", "SEGUIMIENTO"])
stacked_ps[COMPONENT] = stacked

df_pi  = pd.concat([dfs_ps['OA'], dfs_ps['OC']]).copy()  # full dataset for figure cells

df_OA_ps = encode_and_filter(dfs_ps['OA'], INSTANT_MAP_PS, GROUP_MAP_2, groups=["Sham", "Experimental"])
df_OC_ps = encode_and_filter(dfs_ps['OC'], INSTANT_MAP_PS, GROUP_MAP_2, groups=["Sham", "Experimental"])

arrays_OA_ps = prepare_post_fu_arrays(df_OA_ps)
arrays_OC_ps = prepare_post_fu_arrays(df_OC_ps)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 12 — Build & sample: Eyes Open POST->SEG       ║
# ╚══════════════════════════════════════════════════════╝

eo_post_fu_lmm = build_post_fu_lmm(arrays_OA_ps)

with eo_post_fu_lmm:
    eo_post_fu_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 13 — Build & sample: Eyes Closed POST->SEG     ║
# ╚══════════════════════════════════════════════════════╝

ec_post_fu_lmm = build_post_fu_lmm(arrays_OC_ps)

with ec_post_fu_lmm:
    ec_post_fu_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 14 — POST->SEG results                         ║
# ╚══════════════════════════════════════════════════════╝

run_and_print_bayesian_lmm_hdi(eo_post_fu_trace, "Eyes Open (OA)")
run_and_print_bayesian_lmm_hdi(ec_post_fu_trace, "Eyes Closed (OC)")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 15 — Extract POST->SEG posteriors              ║
# ╚══════════════════════════════════════════════════════╝

post_fu_eo = extract_post_fu_posteriors(eo_post_fu_trace)
sham_post_eo_fu, exp_post_eo_fu = post_fu_eo["sham_post"], post_fu_eo["exp_post"]
sham_fu_eo,      exp_fu_eo      = post_fu_eo["sham_fu"],   post_fu_eo["exp_fu"]

post_fu_ec = extract_post_fu_posteriors(ec_post_fu_trace)
sham_post_ec_fu, exp_post_ec_fu = post_fu_ec["sham_post"], post_fu_ec["exp_post"]
sham_fu_ec,      exp_fu_ec      = post_fu_ec["sham_fu"],   post_fu_ec["exp_fu"]

post_fu_posteriors[COMPONENT] = {
    "Eyes Open":   {"sham_post": sham_post_eo_fu, "exp_post": exp_post_eo_fu,
                    "sham_fu":   sham_fu_eo,       "exp_fu":   exp_fu_eo},
    "Eyes Closed": {"sham_post": sham_post_ec_fu, "exp_post": exp_post_ec_fu,
                    "sham_fu":   sham_fu_ec,       "exp_fu":   exp_fu_ec},
}

# Aperiodic (Ξ)

In [ ]:
COMPONENT   = "xi"
channels    = COMPONENTS[COMPONENT]["channels"]
eeg_regions = COMPONENTS[COMPONENT]["regions"]
df_all      = load_data(COMPONENT)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 6 — PRE->POST: pipeline & array preparation    ║
# ╚══════════════════════════════════════════════════════╝

INSTANT_MAP_PP = {"PRE": 0, "POST": 1}
GROUP_MAP_3    = {"Control": 0, "Experimental": 1, "Sham": 2}  # 3-group model

dfs_pp, stacked   = build_pipeline(df_all, channels, instants=["PRE", "POST"])
stacked_pp[COMPONENT] = stacked

df_OA_pp = encode_and_filter(dfs_pp['OA'], INSTANT_MAP_PP, GROUP_MAP_3)
df_OC_pp = encode_and_filter(dfs_pp['OC'], INSTANT_MAP_PP, GROUP_MAP_3)

# Optional sanity check:
# print(df_OA_pp.head(10)); print(df_OA_pp.groupby('id').size().unique())

arrays_OA_pp = prepare_pre_post_arrays(df_OA_pp)
arrays_OC_pp = prepare_pre_post_arrays(df_OC_pp)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 7 — Build & sample: Eyes Open PRE->POST        ║
# ╚══════════════════════════════════════════════════════╝

eo_pre_post_lmm = build_pre_post_lmm(arrays_OA_pp)

with eo_pre_post_lmm:
    eo_pre_post_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — Build & sample: Eyes Closed PRE->POST      ║
# ╚══════════════════════════════════════════════════════╝

ec_pre_post_lmm = build_pre_post_lmm(arrays_OC_pp)

with ec_pre_post_lmm:
    ec_pre_post_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 9 — PRE->POST results                          ║
# ╚══════════════════════════════════════════════════════╝

run_and_print_bayesian_lmm_hdi(eo_pre_post_trace, "Eyes Open (OA)")
run_and_print_bayesian_lmm_hdi(ec_pre_post_trace, "Eyes Closed (OC)")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 10 — Extract PRE->POST posteriors              ║
# ╚══════════════════════════════════════════════════════╝

post_eo = extract_pre_post_posteriors(eo_pre_post_trace)
ctrl_pre_eo,  exp_pre_eo,  sham_pre_eo  = post_eo["ctrl_pre"],  post_eo["exp_pre"],  post_eo["sham_pre"]
ctrl_post_eo, exp_post_eo, sham_post_eo = post_eo["ctrl_post"], post_eo["exp_post"], post_eo["sham_post"]

post_ec = extract_pre_post_posteriors(ec_pre_post_trace)
ctrl_pre_ec,  exp_pre_ec,  sham_pre_ec  = post_ec["ctrl_pre"],  post_ec["exp_pre"],  post_ec["sham_pre"]
ctrl_post_ec, exp_post_ec, sham_post_ec = post_ec["ctrl_post"], post_ec["exp_post"], post_ec["sham_post"]

pre_post_posteriors[COMPONENT] = {
    "Eyes Open":   {"ctrl_pre": ctrl_pre_eo,  "exp_pre": exp_pre_eo,  "sham_pre": sham_pre_eo,
                    "ctrl_post": ctrl_post_eo, "exp_post": exp_post_eo, "sham_post": sham_post_eo},
    "Eyes Closed": {"ctrl_pre": ctrl_pre_ec,  "exp_pre": exp_pre_ec,  "sham_pre": sham_pre_ec,
                    "ctrl_post": ctrl_post_ec, "exp_post": exp_post_ec, "sham_post": sham_post_ec},
}

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 11 — POST->SEG: pipeline & array preparation   ║
# ╚══════════════════════════════════════════════════════╝

INSTANT_MAP_PS = {"POST": 0, "SEGUIMIENTO": 1}
GROUP_MAP_2    = {"Sham": 0, "Experimental": 1}  # 2-group model

dfs_ps, stacked   = build_pipeline(df_all, channels, instants=["POST", "SEGUIMIENTO"])
stacked_ps[COMPONENT] = stacked

df_xi  = pd.concat([dfs_ps['OA'], dfs_ps['OC']]).copy()  # full dataset for figure cells

df_OA_ps = encode_and_filter(dfs_ps['OA'], INSTANT_MAP_PS, GROUP_MAP_2, groups=["Sham", "Experimental"])
df_OC_ps = encode_and_filter(dfs_ps['OC'], INSTANT_MAP_PS, GROUP_MAP_2, groups=["Sham", "Experimental"])

arrays_OA_ps = prepare_post_fu_arrays(df_OA_ps)
arrays_OC_ps = prepare_post_fu_arrays(df_OC_ps)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 12 — Build & sample: Eyes Open POST->SEG       ║
# ╚══════════════════════════════════════════════════════╝

eo_post_fu_lmm = build_post_fu_lmm(arrays_OA_ps)

with eo_post_fu_lmm:
    eo_post_fu_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 13 — Build & sample: Eyes Closed POST->SEG     ║
# ╚══════════════════════════════════════════════════════╝

ec_post_fu_lmm = build_post_fu_lmm(arrays_OC_ps)

with ec_post_fu_lmm:
    ec_post_fu_trace = pm.sample(
        draws=4000, tune=1000, chains=4,
        target_accept=0.9, return_inferencedata=True, random_seed=42,
    )

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 14 — POST->SEG results                         ║
# ╚══════════════════════════════════════════════════════╝

run_and_print_bayesian_lmm_hdi(eo_post_fu_trace, "Eyes Open (OA)")
run_and_print_bayesian_lmm_hdi(ec_post_fu_trace, "Eyes Closed (OC)")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 15 — Extract POST->SEG posteriors              ║
# ╚══════════════════════════════════════════════════════╝

post_fu_eo = extract_post_fu_posteriors(eo_post_fu_trace)
sham_post_eo_fu, exp_post_eo_fu = post_fu_eo["sham_post"], post_fu_eo["exp_post"]
sham_fu_eo,      exp_fu_eo      = post_fu_eo["sham_fu"],   post_fu_eo["exp_fu"]

post_fu_ec = extract_post_fu_posteriors(ec_post_fu_trace)
sham_post_ec_fu, exp_post_ec_fu = post_fu_ec["sham_post"], post_fu_ec["exp_post"]
sham_fu_ec,      exp_fu_ec      = post_fu_ec["sham_fu"],   post_fu_ec["exp_fu"]

post_fu_posteriors[COMPONENT] = {
    "Eyes Open":   {"sham_post": sham_post_eo_fu, "exp_post": exp_post_eo_fu,
                    "sham_fu":   sham_fu_eo,       "exp_fu":   exp_fu_eo},
    "Eyes Closed": {"sham_post": sham_post_ec_fu, "exp_post": exp_post_ec_fu,
                    "sham_fu":   sham_fu_ec,       "exp_fu":   exp_fu_ec},
}

# Figures

In [ ]:
def plot_eeg_posteriors(posteriors, groups, instant_labels, title, filename,
                        bw=0.25, figsize=(21, 9)):

    COLOR_CTRL   = "#2E5FA3"
    COLOR_SHAM   = "#555555"
    COLOR_EXP    = "#B02020"
    GROUP_COLORS = {"Control": COLOR_CTRL, "Sham": COLOR_SHAM, "Experimental": COLOR_EXP}

    conditions  = ["Eyes Open", "Eyes Closed"]
    cond_labels = {"Eyes Open": "Eyes Open (EO)", "Eyes Closed": "Eyes Closed (EC)"}
    components  = ["pi", "xi"]
    panel_text  = {
        "pi": r"Panel A: Periodic $(\Pi)$",
        "xi": r"Panel B: Aperiodic $(\Xi)$",
    }
    col_map = [(0, "pi", 0), (1, "pi", 1), (2, "xi", 0), (3, "xi", 1)]

    # Replaces plt.subplots(2, 4) with a GridSpec that includes a spacer column
    fig = plt.figure(figsize=figsize, facecolor="white")
    gs  = GridSpec(2, 5, figure=fig,
                   width_ratios=[1, 1, 0.15, 1, 1],  # col 2 = spacer between panels
                   hspace=0.12, wspace=0.08)

    # Actual gs column mapping: 0,1 -> Panel A | 2 -> spacer | 3,4 -> Panel B
    gs_cols = [0, 1, 3, 4]  # logical col 0->1->2->3 maps to gs col 0->1->3->4
    axes = {
        (r, lc): fig.add_subplot(gs[r, gc])
        for r in range(2)
        for lc, gc in enumerate(gs_cols)
    }

    # ── X and Y axis ranges, computed per panel/component ────────────────────
    x_grids_panel, x_ranges_panel, y_maxes_panel = {}, {}, {}
    for comp in components:
        all_db = np.concatenate([
            arr
            for cond in conditions
            for g in groups if g in posteriors[comp][cond]
            for arr in posteriors[comp][cond][g]
        ])
        xmin = np.percentile(all_db, 0.1)  - 0.5
        xmax = np.percentile(all_db, 99.9) + 0.5
        xg   = np.linspace(xmin, xmax, 600)
        x_ranges_panel[comp] = (xmin, xmax)
        x_grids_panel[comp]  = xg
        y_maxes_panel[comp]  = max(
            gaussian_kde(arr, bw_method=bw)(xg).max()
            for cond in conditions
            for g in groups if g in posteriors[comp][cond]
            for arr in posteriors[comp][cond][g]
        )

    # ── Plot ──────────────────────────────────────────────────────────────────
    for row_idx, cond in enumerate(conditions):
        for col_idx, comp, t_idx in col_map:
            ax         = axes[row_idx, col_idx]
            xg         = x_grids_panel[comp]
            xmin, xmax = x_ranges_panel[comp]
            ymax       = round(y_maxes_panel[comp], 1)

            for group in groups:
                if group not in posteriors[comp][cond]:
                    continue
                color          = GROUP_COLORS[group]
                arr_t1, arr_t2 = posteriors[comp][cond][group]
                arr = arr_t1 if t_idx == 0 else arr_t2
                kde = gaussian_kde(arr, bw_method=bw)
                y   = kde(xg)
                ax.plot(xg, y, color=color, linewidth=2)
                ax.fill_between(xg, y, alpha=0.13, color=color)
                ax.axvline(np.mean(arr), color=color, linewidth=1, linestyle="--", alpha=0.6)

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(0, ymax * 1.18)
            ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.1f'))
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=16)

            ax.set_xlabel(r"dB", fontsize=19) if row_idx == 1 else ax.set_xlabel("")
            ax.set_ylabel("Posterior density", fontsize=19, labelpad=10) if col_idx == 0 else ax.set_ylabel("")

            if row_idx == 0:
                ax.set_title(instant_labels[t_idx], fontsize=21, pad=6)
            if col_idx == 0:
                ax.annotate(cond_labels[cond], xy=(-0.35, 0.5), xycoords="axes fraction",
                            fontsize=24, fontweight="bold",
                            va="center", ha="center", rotation=90)

    # ── Hide redundant Y tick labels; X tick numbers visible in both rows ─────
    for r in range(2):
        axes[r, 1].tick_params(labelleft=False)
        axes[r, 3].tick_params(labelleft=False)

    # ── Layout: top margin leaves room for the title hierarchy ────────────────
    plt.tight_layout(rect=[0.06, 0.13, 1.0, 0.78])

    # ── Panel labels and main title ───────────────────────────────────────────
    pos   = {(r, c): axes[r, c].get_position() for r in range(2) for c in range(4)}
    top_y = pos[(0, 0)].y1 + 0.07

    fig.text(pos[(0, 0)].x0, top_y, panel_text["pi"],
             ha="left", va="bottom", fontsize=25, fontweight="bold")
    fig.text(pos[(0, 2)].x0, top_y, panel_text["xi"],
             ha="left", va="bottom", fontsize=25, fontweight="bold")
    fig.text(pos[(0, 0)].x0, top_y + 0.05, title,
             ha="left", va="bottom", fontsize=27, fontweight="bold")

    # ── Footer legend ─────────────────────────────────────────────────────────
    GROUP_LABELS = {"Control": "CTRL", "Sham": "SHAM", "Experimental": "EXP"}
    handles = [
        Line2D([0], [0], color=GROUP_COLORS[g], linewidth=2, label=GROUP_LABELS[g])
        for g in groups
    ] + [
        Line2D([0], [0], color="black", linewidth=1,
               linestyle="--", alpha=0.6, label="Posterior mean"),
    ]
    fig.legend(handles=handles, loc="lower center", fontsize=20,
               prop=FontProperties(size=20, weight="bold"),
               frameon=False, ncol=len(handles),
               bbox_to_anchor=(0.5, -0.05))

    plt.savefig(filename, bbox_inches="tight", dpi=300)
    plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 21 — Figure: PRE->POST                         ║
# ╚══════════════════════════════════════════════════════╝

posteriors_pp = {
    comp: {
        "Eyes Open": {
            "Control":      (pre_post_posteriors[comp]["Eyes Open"]["ctrl_pre"],
                             pre_post_posteriors[comp]["Eyes Open"]["ctrl_post"]),
            "Sham":         (pre_post_posteriors[comp]["Eyes Open"]["sham_pre"],
                             pre_post_posteriors[comp]["Eyes Open"]["sham_post"]),
            "Experimental": (pre_post_posteriors[comp]["Eyes Open"]["exp_pre"],
                             pre_post_posteriors[comp]["Eyes Open"]["exp_post"]),
        },
        "Eyes Closed": {
            "Control":      (pre_post_posteriors[comp]["Eyes Closed"]["ctrl_pre"],
                             pre_post_posteriors[comp]["Eyes Closed"]["ctrl_post"]),
            "Sham":         (pre_post_posteriors[comp]["Eyes Closed"]["sham_pre"],
                             pre_post_posteriors[comp]["Eyes Closed"]["sham_post"]),
            "Experimental": (pre_post_posteriors[comp]["Eyes Closed"]["exp_pre"],
                             pre_post_posteriors[comp]["Eyes Closed"]["exp_post"]),
        },
    }
    for comp in ["pi", "xi"]
}

plot_eeg_posteriors(
    posteriors     = posteriors_pp,
    groups         = ["Control", "Sham", "Experimental"],
    instant_labels = ["PRE", "POST"],
    title          = r"Oscillatory Energy Posterior Distributions (PRE$\rightarrow$POST)",
    filename       = "xipi_pre_post_posteriors.pdf",
)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 22 — Figure: POST->FU                          ║
# ╚══════════════════════════════════════════════════════╝

posteriors_ps = {
    comp: {
        "Eyes Open": {
            "Sham":         (post_fu_posteriors[comp]["Eyes Open"]["sham_post"],
                             post_fu_posteriors[comp]["Eyes Open"]["sham_fu"]),
            "Experimental": (post_fu_posteriors[comp]["Eyes Open"]["exp_post"],
                             post_fu_posteriors[comp]["Eyes Open"]["exp_fu"]),
        },
        "Eyes Closed": {
            "Sham":         (post_fu_posteriors[comp]["Eyes Closed"]["sham_post"],
                             post_fu_posteriors[comp]["Eyes Closed"]["sham_fu"]),
            "Experimental": (post_fu_posteriors[comp]["Eyes Closed"]["exp_post"],
                             post_fu_posteriors[comp]["Eyes Closed"]["exp_fu"]),
        },
    }
    for comp in ["pi", "xi"]
}

plot_eeg_posteriors(
    posteriors     = posteriors_ps,
    groups         = ["Sham", "Experimental"],
    instant_labels = ["POST", "FU"],
    title          = r"Oscillatory Energy Posterior Distributions (POST$\rightarrow$FU)",
    filename       = "xipi_post_fu_posteriors.pdf",
)